In [ ]:
# En Google Colab: monta tu Drive y ajusta las rutas de abajo.
# En entorno local: esta celda se puede omitir.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import os
if 'IN_COLAB' in dir() and IN_COLAB:
    os.chdir("/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises")
# En entorno local el directorio de trabajo ya es el correcto.


In [ ]:
import sys, os
# Agrega el directorio padre (donde está src/) al path
parent = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent not in sys.path:
    sys.path.insert(0, parent)


In [ ]:
"""
Taller: Internals de un LLM estilo LLaMA
=================================================================
Instrucciones:
  - Busca los bloques marcados con TODO y completa el código.
  - Cada sección tiene una celda de verificación al final.
  - No modifiques nada fuera de los bloques TODO.

Secciones:
  2. Weight tying — parámetros compartidos
"""

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [ ]:
# ===========================================================================
# SECCIÓN 2 — Weight tying
# ===========================================================================
# El embedding (vocab → d_model) y el lm_head (d_model → vocab) comparten
# el mismo tensor de pesos. Esto reduce parámetros y mejora generalización.
#
# Pregunta: ¿cuántos parámetros se ahorran con weight tying?
# Respuesta: vocab_size * d_model  (el tamaño exacto de la matriz embed/lm_head)
# ===========================================================================

def count_parameters(model: nn.Module) -> int:
    # TODO 2.1 — Retorna el número de parámetros únicos (no contar tensores compartidos)
    # Usamos id() del data_ptr para detectar tensores que comparten memoria.
    seen = set()
    total = 0
    for p in model.parameters():
        if p.data_ptr() not in seen:
            seen.add(p.data_ptr())
            total += p.numel()
    return total


def build_model_no_tying(config: ModelConfig) -> MiniLLaMA:
    # TODO 2.2 — Construye un MiniLLaMA y ROMPE el weight tying.
    # MiniLLaMA.__init__ ya hace: self.lm_head.weight = self.embed.weight
    # Lo deshacemos reemplazando lm_head por un Linear independiente.
    model = MiniLLaMA(config)
    model.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
    nn.init.normal_(model.lm_head.weight, mean=0.0, std=0.02)
    return model


# ── Verificación 2 ──────────────────────────────────────────────────────────
def verify_section2():
    print("=" * 55)
    print("VERIFICACIÓN 2 — Weight tying")
    print("=" * 55)
    cfg = ModelConfig(vocab_size=256, d_model=32, n_heads=4,
                      n_kv_heads=2, d_ff=64, max_seq_len=16)

    model_tied    = MiniLLaMA(cfg)
    model_no_tie  = build_model_no_tying(cfg)

    params_tied   = count_parameters(model_tied)
    params_no_tie = count_parameters(model_no_tie)
    saved         = params_no_tie - params_tied
    expected_save = cfg.vocab_size * cfg.d_model

    print(f"  Params con tying    : {params_tied:,}")
    print(f"  Params sin tying    : {params_no_tie:,}")
    print(f"  Diferencia          : {saved:,}")
    print(f"  Esperado (V×D)      : {expected_save:,}")

    tying_ok = (model_tied.lm_head.weight.data_ptr() ==
                model_tied.embed.weight.data_ptr())
    print(f"  Mismo tensor (tied) : {tying_ok}")
    no_tie_ok = (model_no_tie.lm_head.weight.data_ptr() !=
                 model_no_tie.embed.weight.data_ptr())
    print(f"  Tensor distinto (no tied): {no_tie_ok}")

    if saved == expected_save and tying_ok and no_tie_ok:
        print("\n  ✓ Sección 2 correcta")
    else:
        print("\n  ✗ Revisa tu implementación")


In [ ]:
verify_section2()